In [ ]:
import pandas as pd
sitefinity = pd.read_csv('Sitefinity.csv')
reach = pd.read_csv('reach.csv')

reachList = [i.lower() for i in reach["From REACH"]]
sitefinityList = [i.lower() for i in sitefinity["Sitefinity"]]

sitefinityNotInREACH = []
reachNotInSitefinity = []

for i in sitefinityList:
    if i not in reachList:
        sitefinityNotInREACH.append(i)

for i in reachList:
    if i not in sitefinityList:
        reachNotInSitefinity.append(i)

df = pd.DataFrame(reachNotInSitefinity, columns=["title"])
df.to_csv("REACHH.csv", index=False)

In [ ]:
# Generate collection - link varient
month = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
year = ["2020", "2021", "2022", "2023", "2024"]

for i in year:
    for x in month:
        schema = {
            "version": "0.1.0",
            "layout": "link",
            "page": {
              "title": f"Better Cities {x[0].upper()}{x[1:]} {i}",
              "ref": f"/research-publications/publications-library/{i}-{x}",
              "category": "Better Cities",
              "date": f"1 {x[0].upper()}{x[1:]} {i}"
            },
            "content": []
        }
    
        # Create the directory if it doesn't exist
        os.makedirs("docs", exist_ok=True)
        
        # Create the file path using os.path.join
        file_path = os.path.join("docs", f"{i}-{x}.json")
        
        # Add index to name to prevent duplication
        if os.path.exists(file_path):
            file_path = os.path.join("docs", f"{i}-{x}.json")
        
        # Create json file
        with open(f"{file_path}", 'w+', encoding='utf-8') as f:
            json.dump(schema, f, ensure_ascii=False, indent=2)

In [ ]:
import requests
from requests.auth import HTTPBasicAuth
from urllib.parse import urljoin, urlparse
from dotenv import load_dotenv
import os
import json
from bs4 import BeautifulSoup
from requests.exceptions import SSLError
from urllib3.exceptions import NewConnectionError
import ssl

########################################
#             Side Quest #4            #
########################################

# Load the .env file
load_dotenv()
# Extract password
password = os.getenv("IPOS")

domain = "https://staging.d3t3m6no0k8kp4.amplifyapp.com"
url = domain + "/sitemap.json"

# Get sitemap
page = requests.get(url, auth=HTTPBasicAuth('user', password))
pageJson = json.loads(page.text)

path = []
counter = 0
# Exclude all collection as the children pages are already on the sitemap
collection = []

# Process sitemap up to level 4
for index, value in enumerate(pageJson['children']):
    try:
        if pageJson['children'][index]["layout"] == "collection":
            collectiona.append(pageJson['children'][index]["permalink"].split("/")[-1])
        if 'ref' in pageJson['children'][index]:
            path.append(pageJson['children'][index]['ref'])
            collection.append(pageJson["layout"])
        else:
            path.append(pageJson['children'][index]['permalink'])
        if 'children' in pageJson['children'][index]:
            for i in range(0, len(pageJson['children'][index]['children'])):
                if pageJson['children'][index]["layout"] == "collection":
                    collection.append(pageJson['children'][index]["layout"])
                if 'ref' in pageJson['children'][index]['children'][i]:
                    path.append(pageJson['children'][index]['children'][i]['ref'])
                else:
                    path.append(pageJson['children'][index]['children'][i]['permalink'])
                if 'children' in pageJson['children'][index]['children'][i]:
                    for c in range(0, len(pageJson['children'][index]['children'][i]['children'])):
                        if pageJson['children'][index]['children'][i]["layout"] == "collection":
                            temp = pageJson['children'][index]['children'][i]["permalink"].split("/")[-1]
                            if temp not in collection:
                                collection.append(temp)
                        if 'ref' in pageJson['children'][index]['children'][i]['children'][c]:
                            path.append(pageJson['children'][index]['children'][i]['children'][c]['ref'])
                        else:
                            path.append(pageJson['children'][index]['children'][i]['children'][c]['permalink'])
                        if 'children' in pageJson['children'][index]['children'][i]['children'][c]:
                            for y in range(0, len(pageJson['children'][index]['children'][i]['children'][c]['children'])):
                                if pageJson['children'][index]['children'][i]['children'][c]['layout'] == "collection":
                                    temp = pageJson['children'][index]['children'][i]['children'][c]['permalink'].split("/")[-1]
                                    if temp not in collection:
                                        collection.append(temp)
                                if 'ref' in pageJson['children'][index]['children'][i]['children'][c]['children'][y]:
                                    path.append(pageJson['children'][index]['children'][i]['children'][c]['children'][y]['ref'])
                                else:
                                    path.append(pageJson['children'][index]['children'][i]['children'][c]['children'][y]['permalink'])
    except Exception as e:
        pass
print(len(path), " pages")
print(collection)

# Loop through all pages found in sitemap
for i, v in enumerate(path):
    internalLinks = []

    # Separate workflow if its a external link
    if "https://" in v or "http://" in v or "www." in v:
        try:
            resp = requests.get(i, allow_redirects=True)
            if resp.status_code == 404:
                print("Staging:", v, "404!")
        except NewConnectionError:
            print("Staging:", v, "404!")
        except requests.exceptions.RequestException as e:
            print("Staging:", v, "404!")

    # Seperate workflow if its a internal link
    else:
        url = domain + v
        # Request header data of staging URL
        page = requests.head(url, auth=HTTPBasicAuth('user', password), allow_redirects=True)

        # Check if it redirects to a 404 page
        if page.url != domain + "/404.html":
            # Ignore files and images - they are covered in the workflow below
            if "/files" in v or "/images" in v:
                continue
            elif "/files" not in v or "/images" not in v:
                # Proceed if the filepath is not a collection
                if v.split("/")[-1] not in collection:
                    # Get the contents of the staging URL in HTML
                    page = requests.get(url, auth=HTTPBasicAuth('user', password), allow_redirects=True)
                    soup = BeautifulSoup(page.content, "html.parser")

                    # Try and find all links within available div classes and append to links list
                    # New classes will be iteratively added
                    try:
                        print(url)
                        content = soup.find("div", class_ = "col-span-12 flex flex-col gap-16 lg:col-span-9 lg:mr-24")
                        links = content.find_all("a")
                    except AttributeError:
                        try:
                            print(url)
                            content = soup.find("div", class_ = "grid grid-cols-1 gap-10 md:gap-7 lg:gap-x-16 lg:gap-y-12")
                            links = content.find_all("a")
                        except AttributeError:
                            try:
                                print(url)
                                content = soup.find("div", class_ = "col-span-12 flex flex-col gap-16 max-w-[54rem]")
                                links = content.find_all("a")
                            except AttributeError:
                                print(url)
                                content = soup.find("div", class_ = "mx-auto grid max-w-screen-xl grid-cols-12 px-6 py-12 md:px-10 md:py-16 lg:gap-6 xl:gap-10")
                                links = content.find_all("a")
                                
                    # Loop through all links found in the staging URL
                    for index, value in enumerate(links):
                        try:
                            # Separate workflow if its a external link
                            if "https://" in value["href"] or "http://" in value["href"]:
                                try:
                                    resp = requests.get(value["href"], allow_redirects=True)
                                    if resp.status_code == 404:
                                        print("Staging:", url, "href:", value["href"], "404!")
                                except NewConnectionError:
                                    print("Staging:", url, "href:", value["href"], "404!")
                                except requests.exceptions.RequestException as e:
                                    print("Staging:", url, "href:", value["href"], "404!")
                            # Check if it contains a empty href
                            if "undefined" in value["href"]:
                                print("Staging:", domain+v, "Empty href!", "404!")
                                continue
                            # Exclude all links mail and telephone href
                            if value["href"][0] != "#" and "mailto:" not in value["href"] and "tel:" not in value["href"]:
                                page = requests.head(value["href"] if "https" in value["href"] or "http" in value["href"] else domain + value["href"], auth=HTTPBasicAuth('user', password), allow_redirects=True)
                                # Checks if it leads to error pages - Good to know, but it shouldnt be our problem
                                # You can choose to ignore this output or inform your agency
                                if "PageNotFound" in page.url or page.url == domain + "/404.html" or "closepage" in page.url or page.url == domain + "/undefined":
                                    print("Staging:", domain+v, "href:", value["href"], "Redirected:", page.url, "404!")
                                # If it continue if it doesnt lead to a 404 page
                                if page.url != domain + "/404.html":
                                    print(url)
                                if page.url == domain + "/404.html":
                                    print("Staging:", url, "href:", value["href"], "Page 404!")
                            else:
                                continue
                        # Catch generic site errors
                        except SSLError:
                            print("Staging:", domain+v, "href:", value["href"], page.url, "404!")
                        except requests.exceptions.Timeout:
                            print("Staging:",domain+v, "href:", value["href"], page.url, "404!")
                        except ssl.SSLCertVerificationError as e:
                            print("Staging:",domain+v, "href:", value["href"], page.url, "404!")
                        except Exception as e:
                            print(url, value["href"], e)
        else:
            print("Staging:", url, "href:", value["href"], "Page 404!")

index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
content
content
content
content
content
content
content
content
content
content
content
content
content
content
content
index
index
index
index
index
index
index
index
index
index
index
index
index
index
content
content
content
index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
index
content
content
content
content
content
content
310  pages
['news-collection', 'publications']
https://staging.d3t3m6no0k8kp4.amplifyapp.com/about-IP
https://staging.d3t3m6no0k8kp4.amplifyapp.com/about-IP
https://staging.d3t3m6no0k8kp4.amplifyapp.com/about-IP
https://staging.d3t3m6no0k8kp4.amplifyapp.com/about-IP
https://staging.d3t3m6no0k8kp4.amplifyapp.com/about-IP
https://staging.d3t3m6no0k8kp4.amplifyapp.com/about-IP
https://staging.d3t3m6no0k8kp4.amplifyapp

In [ ]:
path

In [ ]:
import pymupdf
import pathlib

# read HTML file
HTML = pathlib.Path("https://www.csa.gov.sg/alerts-advisories/security-bulletins/2024/sb-2024-049").read_bytes().decode()

story = pymupdf.Story(html=HTML)  # interpret it by the Story object
writer = pymupdf.DocumentWriter("output.pdf")
mediabox = pymupdf.paper_rect("a6")  # choose a small paper size
where = mediabox + (36, 36, -36, -36)  # leave 1/2 inch borders

more = True
while more:  # write on one or more PDF pages as required
 dev = writer.begin_page(mediabox)  # tell the writer our page size
 more, filled = story.place(where)  # compute layout
 story.draw(dev)  # write to the page
 writer.end_page()  # finish page
writer.close()  # close the writer